# Milestone 2 – tuning GAE and VGAE on the HeaRT split

**Team TCM Dynamics**

Same effort as SEAL: 6 configurations per model and dataset. Each configuration is trained once and scored on the **validation** edges; the best one (AUC on easy negatives, MRR on HeaRT negatives) is then re-run with seeds 0–9 and scored on validation and test. The test edges are never used to choose.

Code: `src/tune_m2.py`. Output: `results/m2/raw_tuning.csv`. Use a GPU runtime (Runtime → Change runtime type → GPU).

## 1. Setup

In [ ]:
!pip -q install torch_geometric

import os, sys
if not os.path.exists("gnn-link-prediction"):
    !git clone -q https://github.com/dhanishta-r/gnn-link-prediction.git
os.chdir("gnn-link-prediction")
sys.path.insert(0, "src")

import torch
print("GPU available:", torch.cuda.is_available())

## 2. Run the tuning

In [ ]:
from tune_m2 import tune, DEFAULT_OUT, GRID, config_name

print("Configurations per model:")
for c in GRID:
    print("  ", config_name(c))
if os.path.exists(DEFAULT_OUT):
    os.remove(DEFAULT_OUT)
tune(datasets=("cora", "citeseer", "pubmed"), seeds=range(10), out_csv=DEFAULT_OUT, root="data")

## 3. All validation scores of the tuning step

In [ ]:
import pandas as pd
t = pd.read_csv(DEFAULT_OUT)
v = t[(t.notes == "tuning")]
pd.concat([
    v[v.setting == "easy"].pivot_table(index=["dataset", "model", "config"], values="AUC").rename(columns={"AUC": "valid easy AUC"}),
    v[v.setting == "heart"].pivot_table(index=["dataset", "model", "config"], values="MRR").rename(columns={"MRR": "valid HeaRT MRR"}),
], axis=1).round(2)

## 4. Save the results

This downloads `raw_tuning.csv`. Send it back, or put it in `results/m2/` and push.

In [ ]:
try:
    from google.colab import files
    files.download(DEFAULT_OUT)
except ImportError:
    print("Not on Colab. The file is at", os.path.abspath(DEFAULT_OUT))